In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path(r"D:\1000_DataScience_MachineLearning\1000_ML_Projects\Attention_Prototype\data\raw_csv\baywa_joined_tables.csv")

df = pd.read_csv(
    data_path,
    sep=";",
    encoding="cp1252",
    engine="python",
    dtype="str",
    na_filter=False,
)

df.columns.tolist(), df.shape

(['WRBTR_s',
  'MWSKZ_s',
  'XBLNR',
  'WAERS',
  'SGTXT',
  'BKTXT',
  'HKONT',
  'BUDAT_year',
  'BUDAT_month',
  'BUDAT_day',
  'BLDAT_year',
  'BLDAT_month',
  'BLDAT_day'],
 (191372, 13))

In [2]:
target_col = "HKONT"

num_cols = ["WRBTR_s"]

cat_cols = [
    "MWSKZ_s",
    "WAERS",
    "BUDAT_year",
    "BUDAT_month",
    "BUDAT_day",
    "BLDAT_year",
    "BLDAT_month",
    "BLDAT_day",
]

# cat_cols = [
#     "MWSKZ_s",
#     "WAERS",
# ]

work_df = df[num_cols + cat_cols + [target_col]].copy()

# einfache Bereinigung
for c in num_cols:
    work_df[c] = (
        work_df[c]
        .str.replace(",", ".", regex=False)
        .replace("", "0")
        .astype(float)
    )

for c in cat_cols + [target_col]:
    work_df[c] = work_df[c].fillna("").replace("", "__MISSING__").astype(str)

work_df.shape, work_df[target_col].nunique(), work_df[target_col].value_counts().head(20)

((191372, 10),
 184,
 HKONT
 0059000500    12041
 0062355090    11736
 0062355040    11657
 0063890050    10604
 0062350090    10533
 0062300100     6607
 0062355020     6359
 0062200200     5930
 0062600001     5458
 0062600000     4423
 0062250000     4362
 0062200000     4276
 0062100030     4204
 0062450000     4097
 0062156010     4064
 0062202000     3837
 0063890030     3700
 0062210010     3626
 0062400800     3364
 0062500010     3246
 Name: count, dtype: int64)

In [3]:
TOP_N = 20

top_classes = work_df[target_col].value_counts().head(TOP_N).index.tolist()

proto_df = work_df[work_df[target_col].isin(top_classes)].copy()

proto_df.shape, proto_df[target_col].nunique()

((124124, 10), 20)

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
proto_df["target"] = le.fit_transform(proto_df[target_col])

n_classes = len(le.classes_)
assert n_classes == TOP_N

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    proto_df,
    test_size=0.2,
    stratify=proto_df["target"],
    random_state=42,
)

train_df.shape, test_df.shape

((99299, 11), (24825, 11))

### numerical stabilization of WRBTR (big range)

In [6]:
# nach dem train/test split
train_df = train_df.copy()
test_df = test_df.copy()

lo, hi = train_df["WRBTR_s"].quantile([0.001, 0.999])

train_df["WRBTR_s"] = train_df["WRBTR_s"].clip(lo, hi)
test_df["WRBTR_s"] = test_df["WRBTR_s"].clip(lo, hi)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_df["WRBTR_s"] = scaler.fit_transform(train_df[["WRBTR_s"]])
test_df["WRBTR_s"] = scaler.transform(test_df[["WRBTR_s"]])


## Schritt 2: Kategorie-Vokabulare bauen

Jede kategoriale Spalte bekommt eigenes Lookup.

In [7]:
cat_vocab = {}

for col in cat_cols:
    values = sorted(train_df[col].unique().tolist())
    cat_vocab[col] = {"__UNK__": 0}
    cat_vocab[col].update({v: i + 1 for i, v in enumerate(values)})

{k: len(v) for k, v in cat_vocab.items()}

{'MWSKZ_s': 59,
 'WAERS': 10,
 'BUDAT_year': 3,
 'BUDAT_month': 14,
 'BUDAT_day': 33,
 'BLDAT_year': 10,
 'BLDAT_month': 14,
 'BLDAT_day': 33}

## Schritt 3: Dataset Klasse

Erster Modellcode

In [8]:
import torch
from torch.utils.data import Dataset

class SAPTabularDataset(Dataset):
    def __init__(self, df, cat_cols, num_cols, target_col, cat_vocab):
        self.df = df.reset_index(drop=True)
        self.cat_cols = cat_cols
        self.num_cols = num_cols
        self.target_col = target_col
        self.cat_vocab = cat_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        cat_tokens = []
        for col in self.cat_cols:
            cat_tokens.append(self.cat_vocab[col].get(row[col], 0))

        num_values = torch.tensor(
            [float(row[col]) for col in self.num_cols],
            dtype=torch.float32
        )

        target = torch.tensor(row[self.target_col], dtype=torch.long)

        return {
            "cat": torch.tensor(cat_tokens, dtype=torch.long),
            "num": num_values,
            "target": target,
        }

## Schritt 4: Instanziieren

In [9]:
num_cols = ["WRBTR_s"]

train_ds = SAPTabularDataset(
    train_df,
    cat_cols=cat_cols,
    num_cols=num_cols,
    target_col="target",
    cat_vocab=cat_vocab,
)

test_ds = SAPTabularDataset(
    test_df,
    cat_cols=cat_cols,
    num_cols=num_cols,
    target_col="target",
    cat_vocab=cat_vocab,
)

## Schritt 5: Sanity Check

In [10]:
sample = train_ds[0]

sample["cat"].shape, sample["num"].shape, sample["target"]

(torch.Size([8]), torch.Size([1]), tensor(7))

In [11]:
# Expected:

# cat: torch.Size([8])
# num: torch.Size([1])
# target: scalar

WRBTR_s sollte standardisiert werden. Sonst kann der numerische Token sehr ungünstig skaliert sein. 

In [12]:
import numpy as np

print(proto_df["WRBTR_s"].describe())
print(np.isfinite(proto_df["WRBTR_s"]).value_counts())

count    124124.000000
mean        525.577089
std        3067.505193
min           0.010000
25%          13.490000
50%          80.005000
75%         276.000000
max      466101.710000
Name: WRBTR_s, dtype: float64
WRBTR_s
True    124124
Name: count, dtype: int64


### Modell-Definition

In [13]:
import torch
import torch.nn as nn

class TabularTransformerClassifier(nn.Module):
    def __init__(
        self,
        cat_vocab,
        num_numeric=1,
        n_classes=20,
        d_model=64,
        n_heads=4,
        n_layers=2,
        ff_dim=128,
        dropout=0.1,
    ):
        super().__init__()

        self.cat_cols = list(cat_vocab.keys())
        self.n_cat = len(self.cat_cols)

        # eigene Embedding-Tabelle pro kategorialem Feature
        self.cat_embeddings = nn.ModuleDict({
            col: nn.Embedding(len(cat_vocab[col]), d_model)
            for col in self.cat_cols
        })

        # numerische Features -> Token
        self.num_projection = nn.Linear(num_numeric, d_model)

        # CLS Token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # Positions-Embeddings
        total_tokens = 1 + 1 + self.n_cat
        self.pos_embedding = nn.Parameter(
            torch.randn(1, total_tokens, d_model)
        )

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
        )

        # Klassifikationskopf
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, cat, num):
        batch_size = cat.shape[0]

        # categorical tokens
        cat_tokens = []
        for i, col in enumerate(self.cat_cols):
            emb = self.cat_embeddings[col](cat[:, i])
            cat_tokens.append(emb)

        cat_tokens = torch.stack(cat_tokens, dim=1)

        # numeric token
        num_token = self.num_projection(num).unsqueeze(1)

        # CLS
        cls = self.cls_token.expand(batch_size, -1, -1)

        # alles zusammen
        x = torch.cat([cls, num_token, cat_tokens], dim=1)

        # positions
        x = x + self.pos_embedding

        # encoder
        x = self.encoder(x)

        # CLS Output
        cls_out = x[:, 0, :]

        logits = self.classifier(cls_out)

        return logits

## Test

In [14]:
model = TabularTransformerClassifier(
    cat_vocab=cat_vocab,
    num_numeric=1,
    n_classes=n_classes,
)

model

TabularTransformerClassifier(
  (cat_embeddings): ModuleDict(
    (MWSKZ_s): Embedding(59, 64)
    (WAERS): Embedding(10, 64)
    (BUDAT_year): Embedding(3, 64)
    (BUDAT_month): Embedding(14, 64)
    (BUDAT_day): Embedding(33, 64)
    (BLDAT_year): Embedding(10, 64)
    (BLDAT_month): Embedding(14, 64)
    (BLDAT_day): Embedding(33, 64)
  )
  (num_projection): Linear(in_features=1, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    

In [15]:
sample_batch = next(iter(torch.utils.data.DataLoader(train_ds, batch_size=32)))

out = model(
    sample_batch["cat"],
    sample_batch["num"],
)

out.shape

torch.Size([32, 20])

## Training

In [16]:
from torch.utils.data import DataLoader

batch_size = 512

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
)

In [17]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [18]:
model = TabularTransformerClassifier(
    cat_vocab=cat_vocab,
    num_numeric=1,
    n_classes=n_classes,
    d_model=64,
    n_heads=4,
    n_layers=2,
    ff_dim=128,
    dropout=0.1,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

### Eine Epoche:

In [19]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for batch in loader:
        cat = batch["cat"].to(device)
        num = batch["num"].to(device)
        y = batch["target"].to(device)

        optimizer.zero_grad()

        logits = model(cat, num)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # gradient clipping for numerical stability
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_n += y.size(0)

    return total_loss / total_n, total_correct / total_n

### Evaluation

In [20]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for batch in loader:
        cat = batch["cat"].to(device)
        num = batch["num"].to(device)
        y = batch["target"].to(device)

        logits = model(cat, num)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_n += y.size(0)

    return total_loss / total_n, total_correct / total_n

### Trainings-Schleife:

In [21]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
for epoch in range(1, 16):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}"
    )

epoch=01 train_loss=2.6519 train_acc=0.2410 test_loss=2.1572 test_acc=0.3796
epoch=02 train_loss=2.0335 train_acc=0.4047 test_loss=1.8390 test_acc=0.4492
epoch=03 train_loss=1.8444 train_acc=0.4482 test_loss=1.7351 test_acc=0.4873
epoch=04 train_loss=1.7589 train_acc=0.4747 test_loss=1.6796 test_acc=0.5021
epoch=05 train_loss=1.7115 train_acc=0.4907 test_loss=1.6419 test_acc=0.5137
epoch=06 train_loss=1.6742 train_acc=0.5012 test_loss=1.6053 test_acc=0.5206
epoch=07 train_loss=1.6448 train_acc=0.5093 test_loss=1.5815 test_acc=0.5235
epoch=08 train_loss=1.6193 train_acc=0.5161 test_loss=1.5623 test_acc=0.5276
epoch=09 train_loss=1.5997 train_acc=0.5214 test_loss=1.5428 test_acc=0.5358
epoch=10 train_loss=1.5838 train_acc=0.5260 test_loss=1.5321 test_acc=0.5367
epoch=11 train_loss=1.5709 train_acc=0.5280 test_loss=1.5163 test_acc=0.5402
epoch=12 train_loss=1.5600 train_acc=0.5311 test_loss=1.5063 test_acc=0.5417
epoch=13 train_loss=1.5478 train_acc=0.5349 test_loss=1.4964 test_acc=0.5449

## Base-Line: 

### Mehrheitsklasse

In [28]:
baseline_acc = test_df["target"].value_counts(normalize=True).max()
baseline_acc

np.float64(0.09699899295065459)

### HistGradientBoostingClassifier:

In [29]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

feature_cols = num_cols + cat_cols

X_train = train_df[feature_cols].copy()
y_train = train_df["target"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["target"].copy()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42,
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", hgb),
])

pipe.fit(X_train, y_train)

pred_train = pipe.predict(X_train)
pred_test = pipe.predict(X_test)

print("train_acc:", accuracy_score(y_train, pred_train))
print("test_acc:", accuracy_score(y_test, pred_test))

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Hal9\anaconda3\envs\attention_proto\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\Hal9\anaconda3\envs\attention_proto\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Hal9\anaconda3\envs\attention_proto\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x81 in position 108: invalid start byte


train_acc: 0.7132196698859001
test_acc: 0.6345619335347432


Wir könnten versuchen aus dem Mischfeld XBLNR strukturelle Information zu extrahieren:

In [32]:
df["XBLNR"].nunique()

103182

In [35]:
print(df["XBLNR"].sample(100))

18167          14412145
116999       6000043323
44059      2024-1009647
190877       RA 5262637
140930       2015608787
              ...      
38994          17347490
65221         322689474
152148    76/370/019051
108825        287054736
65355         322691589
Name: XBLNR, Length: 100, dtype: str


103182 verschiedene Werte zeigen an, dass wir hier entweder mit Feature-Engineering arbeiten müssten, was wir nicht wollen, da wir die Transformer nutzen, um zu einem universellen Werkzeug zu gelangen.<br>
Text-Embeddings wären möglich - aber zu diesem Zeitpunkt zu aufwändig - Text-Embeddings sollten wir jetzt vielleicht lieber für richtige Texte SGTXT udn BKTXT nutzen.